# Importing modules and settings

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context

General settings of Scanpy

In [ ]:
sc.settings.figdir = './Preprocessing figures'

In [ ]:
sc.settings.verbosity = 4
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')


In [ ]:
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:azure'], as_cmap = True)

# Declaring the input and output files

In [ ]:
name_of_analysis = 'Serotonergic_data_preprocessed'

In [ ]:
results_file = name_of_analysis+'.h5ad'

In [ ]:
adata = sc.read_10x_mtx(
    "../0 Reads Processing/Sero_matrix/",
    var_names='gene_symbols',
    cache=True)

In [ ]:
adata.var_names_make_unique()

In [ ]:
adata.var_names

In [ ]:
adata.var

In [ ]:
adata.obs

In [ ]:
adata

# Reading the barcodes

In [ ]:
barcodes = pd.read_excel('./barcodes sero pitx-lhxRNAi.xlsx', header = None, index_col = 0)

The barcodes dataframe has to have 96 rows, with the list of barcodes in one column and the names of the different samples. We also need a samples_name that describes the type of samples

In [ ]:
samples_name = 'sample'

In [ ]:
B = 'biological_rep'

In [ ]:
T = 'technical_rep'

In [ ]:
barcodes.rename( columns = {1:'bc', 2: samples_name, 3: B, 4: T}, inplace = True)

In [ ]:
barcodes

There are the following **samples** to be annotated:

In [ ]:
barcodes[samples_name].value_counts()

In [ ]:
samples = barcodes[samples_name].value_counts().index.to_list()

In [ ]:
samples

In [ ]:
B_rep = barcodes[B].value_counts().index.to_list()

In [ ]:
B_rep

In [ ]:
T_rep = barcodes[T].value_counts().index.to_list()

In [ ]:
T_rep

In [ ]:
adata.uns['samples'] = samples

In [ ]:
adata.uns['biological_replicates'] = B_rep

In [ ]:
adata.uns['technical_replicates'] = T_rep

In [ ]:
adata.uns

# Transfering the samples to adata.obs

We want to create a new column in adata.obs with the category of each cell

In [ ]:
adata.obs

The following nested loop does this automatically.  

It loops through each of the samples

For each sample, it establishes a filter in the barcodes dataframe

Then obtains the different barcodes for each into a list

and then loops through the different barcodes of each sample.

In the inner nested loop, there is a filtering of all cells that contain the barcode preceded by "_"

and then it creates the new column and inserts the name of the sample



In [ ]:
for sample_i in samples:
    filt = (barcodes[samples_name] == sample_i)
    li = barcodes[filt]['bc'].to_list()
    for barcode in li:
        cellfilt = adata.obs.index.str.contains("_"+barcode)
        adata.obs.loc[cellfilt, samples_name] = sample_i

In [ ]:
adata.obs

In [ ]:
for B_rep_i in B_rep:
    filt = (barcodes[B] == B_rep_i)
    li = barcodes[filt]['bc'].to_list()
    for barcode in li:
        cellfilt = adata.obs.index.str.contains("_"+barcode)
        adata.obs.loc[cellfilt, B] = B_rep_i

In [ ]:
adata.obs

In [ ]:
for T_rep_i in T_rep:
    filt = (barcodes[T] == T_rep_i)
    li = barcodes[filt]['bc'].to_list()
    for barcode in li:
        cellfilt = adata.obs.index.str.contains("_"+barcode)
        adata.obs.loc[cellfilt, T] = T_rep_i

In [ ]:
adata.obs

# Checking that the samples have been annotated correctly

Introduce a barcode to check and see if all the cells with the barcode have the correct sample annotated

In [ ]:
check = 'TCCTACCAGT'

In [ ]:
cfilt = adata.obs.index.str.contains("_"+check)

In [ ]:
adata.obs[cfilt]

In [ ]:
adata.obs[cfilt].value_counts()

# Annotating the libraries

Annotating the different libraries in adata.obs

Start by giving the names of the libraries in a list

In [ ]:
libraries = ['L43_1', 'L43_2', 'L43_3', 'L43_4', 'L43_5']

The following code will label the libraries as 'Lib1', 'Lib2', etc.

In [ ]:
for lib_i in libraries:
  libfilt = adata.obs.index.str.contains(lib_i)
  adata.obs.loc[libfilt, 'library'] = lib_i

In [ ]:
adata.obs

In [ ]:
c = 1
for lib_i in libraries:
    libfilt = adata.obs.index.str.contains(lib_i)
    adata.obs.loc[libfilt, 'library'] = "Lib"+str(c)
    c += 1
    

In [ ]:
adata.obs

# Checking that the libraries have been annotated correctly

In [ ]:
check2 = 'L43_1'

In [ ]:
c2filt = adata.obs.index.str.contains(check2)

In [ ]:
adata.obs[c2filt]

In [ ]:
adata.obs[c2filt].value_counts()

# Creating a unique id for sample plus library

This cell annotates each cell in a unique sample ID (sample and library)

In [ ]:
adata.obs['sample_bio'] = adata.obs[samples_name] + "_" + adata.obs['biological_rep']

In [ ]:
adata.obs['unique'] = adata.obs[samples_name] + "_" + adata.obs['library']

In [ ]:
adata.obs

# Creating a unique id for sample plus replica

This cell annotates each cell in a unique sample ID (sample and replica)

In [ ]:
adata.obs['sample_rep'] = adata.obs[samples_name] + "_" + adata.obs['biological_rep'] + "_" + adata.obs['technical_rep']

In [ ]:
adata.obs

# Creating a unique id for sample plus library plus replica

This cell annotates each cell in a unique sample ID (sample, library and replica)

In [ ]:
adata.obs['unique_rep'] = adata.obs[samples_name] + "_" + adata.obs['biological_rep'] + "_" + adata.obs['technical_rep'] + "_" + adata.obs['library']

In [ ]:
adata.obs

In [ ]:
adata.obs['unique'].value_counts()

In [ ]:
adata.obs['sample_rep'].value_counts()

In [ ]:
adata.obs['unique_rep'].value_counts()

# Annotating the adata.var dataframe

In [ ]:
ann = pd.read_csv('./20230530_Smed_Rink_Simplified_Annotation_Table.tsv', sep = '\t', index_col = 'gene')

In [ ]:
ann

In [ ]:
ann.columns

In [ ]:
len(ann.columns)

In [ ]:
adata.var

In [ ]:
adata.var = pd.concat([adata.var, ann], axis = 1, join = 'inner')

In [ ]:
adata.var

# Preprocessing

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
sc.pp.filter_cells(adata, min_counts=100)
sc.pp.filter_cells(adata, min_genes= 100) 

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
adata.obs

In [ ]:
adata.obs.groupby('library').describe()

In [ ]:
adata.var

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'],
             jitter=0.4, multi_panel=True, log = True)

In [ ]:
sc.pl.violin(adata, keys = ['n_genes', 'total_counts', 'n_counts'] , groupby = 'library', log = True, jitter = False, multi_panel = True, rotation = 90)

In [ ]:
sc.pl.violin(adata, keys = ['n_genes', 'total_counts', 'n_counts'] , groupby = 'sample', log = True, jitter = False, multi_panel = True, rotation = 90)

In [ ]:
adata

# Matrix slicing

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 900, :]

In [ ]:
adata = adata[adata.obs.total_counts < 1200, :]

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
adata

# Normalization and log transformation

The following 2 functions normalise and log transform the matrix

In [ ]:
sc.pp.normalize_total(adata)

In [ ]:
sc.pp.log1p(adata)

# Selecting highly variable genes

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes = 24000)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.raw = adata

In [ ]:
adata = adata[:, adata.var.highly_variable]

In [ ]:
adata

In [ ]:
adata.raw.var

# Scaling the data

In [ ]:
sc.pp.scale(adata)

# Performing the PCA and kNN analysis

In [ ]:
sc.tl.pca(adata, svd_solver='arpack', n_comps = 150)

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=150, log=True)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=55, n_pcs=95)

In [ ]:
sc.tl.umap(adata, min_dist=0.75, spread = 1.25, alpha = 1, gamma = 1.0)

In [ ]:
sc.pl.umap(adata)

In [ ]:
#This is the figure
fig, axs = plt.subplots(3, 4, figsize = (20, 15))
#Here are the plots that will go into the figure
#Row 0 first row
gene00, name00 = 'h1SMcG0013999', 'piwi+'
gene01, name01 = 'h1SMcG0016003', 'coe+'
gene02, name02 = 'h1SMcG0017844', 'syn3+'
gene03, name03 = 'h1SMcG0013018', 'pc2+'
sc.pl.umap(adata, color= gene00, title = name00+' '+gene00, color_map = umap_cmap, show = False, ax = axs[0, 0])
sc.pl.umap(adata, color= gene01, title = name01+' '+gene01, color_map = umap_cmap, show = False, ax = axs[0, 1])
sc.pl.umap(adata, color= gene02, title = name02+' '+gene02, color_map = umap_cmap, show = False, ax = axs[0, 2])
sc.pl.umap(adata, color= gene03, title = name03+' '+gene03, color_map = umap_cmap, show = False, ax = axs[0, 3])

#Row 1 second row
gene10, name10 = 'h1SMcG0011946', 'vglut+'
gene11, name11 = 'h1SMcG0009545', 'chat+'
gene12, name12 = 'h1SMcG0021560', 'th+'
gene13, name13 = 'h1SMcG0000075', 'tph+'
sc.pl.umap(adata, color= gene10, title = name10+' '+gene10, color_map = umap_cmap, show = False, ax = axs[1, 0])
sc.pl.umap(adata, color= gene11, title = name11+' '+gene11, color_map = umap_cmap, show = False, ax = axs[1, 1])
sc.pl.umap(adata, color= gene12, title = name12+' '+gene12, color_map = umap_cmap, show = False, ax = axs[1, 2])
sc.pl.umap(adata, color= gene13, title = name13+' '+gene13, color_map = umap_cmap, show = False, ax = axs[1, 3])

#Row 2 third row
gene20, name20 = 'h1SMcG0009810', 'gad+'
gene21, name21 = 'h1SMcG0010466', 'glyt+'
gene22, name22 = 'h1SMcG0007800', 'opsin+'
gene23, name23 = 'h1SMcG0019080', 'estrella+'
sc.pl.umap(adata, color= gene20, title = name20+' '+gene20, color_map = umap_cmap, show = False, ax = axs[2, 0])
sc.pl.umap(adata, color= gene21, title = name21+' '+gene21, color_map = umap_cmap, show = False, ax = axs[2, 1])
sc.pl.umap(adata, color= gene22, title = name22+' '+gene22, color_map = umap_cmap, show = False, ax = axs[2, 2])
sc.pl.umap(adata, color= gene23, title = name23+' '+gene23, color_map = umap_cmap, show = False, ax = axs[2, 3])

In [ ]:
#nanos
feature = 'h1SMcG0003273'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 30, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

# Clustering

Indicate how many different resolutions you want to try

In [ ]:
resolutions = [1, 2, 3, 4]

In [ ]:
for i in resolutions:
    sc.tl.leiden(adata, resolution = i, key_added = 'leiden_'+str(i))
    sc.pl.umap(adata, color='leiden_'+str(i))

In [ ]:
leiden_names = adata.obs.columns[adata.obs.columns.str.contains('leiden')].to_list()

In [ ]:
leiden_names

In [ ]:
sc.pl.umap(adata, color=leiden_names, legend_loc = 'on data', legend_fontsize = 10)

In [ ]:
for leiden_i in leiden_names:
    with rc_context({'figure.figsize': (15, 15)}):
        sc.pl.umap(adata, color=leiden_i, legend_loc='on data', title=str(leiden_i), size = 50, frameon=False, 
                   save = '_'+str(leiden_i))

In [ ]:
for leiden_i in leiden_names:
    sc.pl.correlation_matrix(adata, leiden_i, figsize=(25,25),
                            save = '_'+str(leiden_i))

In [ ]:
for leiden_i in leiden_names:
    with rc_context({'figure.figsize': (15, 5)}):
        sc.pl.dendrogram(adata, leiden_i,
                        save = '_'+str(leiden_i))

In [ ]:
for leiden_i in leiden_names:
    sc.tl.rank_genes_groups(adata, leiden_i, method='logreg', key_added = 'rank_genes_groups_logreg_'+str(leiden_i))
    sc.pl.rank_genes_groups(adata, key='rank_genes_groups_logreg_'+str(leiden_i), n_genes = 10, sharey = False)

In [ ]:
for leiden_i in leiden_names:
    sc.tl.rank_genes_groups(adata, leiden_i, method='wilcoxon', key_added = 'rank_genes_groups_wilcox_'+str(leiden_i))
    sc.pl.rank_genes_groups(adata, key='rank_genes_groups_wilcox_'+str(leiden_i), n_genes = 10, sharey = False)

In [ ]:
adata.write(results_file)

# Markers

In [ ]:
#early epidermal progenitors
feature = 'h1SMcG0005537'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#late epidermal progenitors 1/2
feature = 'h1SMcG0008281'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#late epidermal progenitors 2/1
feature = 'h1SMcG0002552'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#epidermis 2
feature = 'h1SMcG0014762'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#epidermis DVb
feature = 'h1SMcG0021341'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#pharynx cell type 
feature = 'h1SMcG0006639'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#otf
feature = 'h1SMcG0002253'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#body wall muscle
feature = 'h1SMcG0020022'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
# pharynx muscle
feature = 'h1SMcG0013876'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#intestinal and DV muscle
feature = 'h1SMcG0001827'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#late muscle
feature = 'h1SMcG0022555'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#early muscle (with progenitors)
feature = 'h1SMcG0020983'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#ldlrr-1
feature = 'h1SMcG0003229'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#aqp
feature = 'h1SMcG0022090'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#pgrn
feature = 'h1SMcG0010631'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#psap
feature = 'h1SMcG0005210'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#pbgd-1
feature = 'h1SMcG0005606'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#phagocytes
feature = 'h1SMcG0018987'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#phagocyte progenitors
feature = 'h1SMcG0008987'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#goblet cells
feature = 'h1SMcG0012530'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#basal cells - goblet cell progenitors
feature = 'h1SMcG0000695'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#slc22a6
feature = 'h1SMcG0015147'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#psd cells
feature = 'h1SMcG0002113'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#protonephridia flame cells
feature = 'h1SMcG0007079'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#cavII-1
feature = 'h1SMcG0020757'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#secretory 1 - lemA-like plass et al
feature = 'h1SMcG0017679'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#secretory 3 glipr-1 #plass et al
feature = 'h1SMcG0000137'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#secretory 4 pdi-1 #plass et al
feature = 'h1SMcG0011317'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#secretory 2 garcia castro et al
feature = 'h1SMcG0021790'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#secretory 3 garcia castro et al
feature = 'h1SMcG0005324'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#secretory 5 garcia castro et al
feature = 'h1SMcG0016901'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])

In [ ]:
#secretory 6 garcia castro et al
feature = 'h1SMcG0015651'
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color= feature, size = 40, cmap = umap_cmap, 
               title = feature + " / " + 
               adata.raw.var.loc[feature]['Preferred_name'] + " / " + 
               adata.raw.var.loc[feature]['ddv6_collapsed'] + " / " +
               adata.raw.var.loc[feature]['gene_Jakke_ver1'])